In [1]:
import polars as pl
import yaml

with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [2]:
DIR = f"{config['dirs']['output']}/{config['city']}_ids_not_replaced"

In [3]:
for i in range(1):
    temp = pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet")

In [5]:
temp

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
"""0000051fc26276495c5a9ab1753a70…",1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,11
"""0000051fc26276495c5a9ab1753a70…",2,2024-10-02 10:23:47 CEST,2024-10-02 10:33:00 CEST,9m 13s,47.661216,19.076262,4.0,1.0,2,11
"""0000051fc26276495c5a9ab1753a70…",3,2024-10-03 10:50:40 CEST,2024-10-03 10:58:29 CEST,7m 49s,47.498402,19.0404,0.0,0.0,2,12
"""0000051fc26276495c5a9ab1753a70…",4,2024-10-04 12:41:20 CEST,2024-10-04 12:49:26 CEST,8m 6s,47.498402,19.0404,0.0,0.0,2,12
"""0000051fc26276495c5a9ab1753a70…",5,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,12
…,…,…,…,…,…,…,…,…,…,…
"""00ffd69f0786f191b711f219fd6605…",381,2024-11-17 18:27:21 CET,2024-11-17 18:37:33 CET,10m 12s,47.606706,19.270747,14.0,12.0,3,4
"""00ffd69f0786f191b711f219fd6605…",382,2024-11-17 21:11:36 CET,2024-11-17 21:26:20 CET,14m 44s,47.606829,19.270937,11.0,6.0,3,4
"""00ffd69f0786f191b711f219fd6605…",383,2024-11-18 04:47:22 CET,2024-11-18 05:13:19 CET,25m 57s,47.60688,19.270975,9.0,2.0,4,4


In [ ]:
import polars_distance as pld
# https://ion-elgreco.github.io/polars-distance/api/dist/#polars_distance.DistancePairWise.haversine

In [ ]:
test = temp.filter(pl.col.device_id.str.starts_with("0000051fc26276495c5a9ab1753a70"))[
    :3
].clone()

In [ ]:
def rg(data: pl.DataFrame) -> pl.DataFrame:
    d = data.with_columns(
        pl.struct("longitude_mean", "latitude_mean")
        .alias("stop")
        .struct.rename_fields(["longitude", "latitude"]),
        pl.struct(pl.col.longitude_mean.mean(), pl.col.latitude_mean.mean())
        .alias("centroid")
        .struct.rename_fields(["longitude", "latitude"]),
    ).with_columns(
        pld.col("centroid").dist.haversine("stop", "km").alias("haversine"),
        pl.col.interval_lenght.dt.total_seconds().sum().alias("N"),
    )
    res = (
        ((d["haversine"] ** 2 * d["interval_lenght"].dt.total_seconds()).sum() / d["N"])
        .sqrt()
        .round(3)[0]
    )
    return data.head(1).with_columns(pl.lit(res).alias("result"))


In [76]:
temp.head(1).with_columns(pl.lit(0).alias("result"))

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,result
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,i32
"""0000051fc26276495c5a9ab1753a70…",1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,11,0


In [ ]:
q = rg(test)
q

8.241283721822473

In [50]:
((q["haversine"] ** 2 * q["interval_lenght"].dt.total_seconds()).sum() / q["N"]).sqrt()

N
f64
8.241284
8.241284
8.241284


In [81]:
temp.group_by("device_id").map_groups(rg)


device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,stop,centroid,haversine,N,result
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,struct[2],struct[2],f64,i64,f64
"""00f70fce9f3bb2942e9238dfd9960b…",1,2023-10-27 02:15:10 CEST,2023-10-27 02:22:04 CEST,6m 54s,47.51374,19.219235,3.0,3.0,3,11,"{19.219235,47.51374}","{19.190824,47.527842}",2.647667,146016,14.244
"""0093107f97719ed33f6e48e030b01d…",1,2024-10-05 02:35:08 CEST,2024-10-05 02:53:35 CEST,18m 27s,47.441729,18.832365,9.0,14.0,4,1,"{18.832365,47.441729}","{18.830269,47.438448}",0.397394,2130,0.514
"""009694d5bbacb1ecb734bb62a9c854…",1,2025-12-23 16:09:02 CET,2025-12-23 16:16:22 CET,7m 20s,48.034006,19.9749,7.0,10.0,2,0,"{19.9749,48.034006}","{19.9749,48.034006}",0.0,440,0.0
"""004c98e5a49ac8a2d7e2515e51373c…",1,2025-08-28 15:19:15 CEST,2025-08-28 15:30:32 CEST,11m 17s,47.495732,19.064934,0.0,6.0,2,9,"{19.064934,47.495732}","{19.092882,47.507695}",2.48538,76444,2.132
"""00b8434096bfdc9ec58857a147bb7b…",1,2023-12-13 09:42:55 CET,2023-12-13 09:52:14 CET,9m 19s,47.441499,19.125149,3.0,1.0,4,3,"{19.125149,47.441499}","{19.128227,47.550818}",12.157867,629884,7.974
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""00599b7d2ee8d06336ca49c98ca5e4…",1,2025-06-22 20:20:57 CEST,2025-06-22 20:34:34 CEST,13m 37s,47.563829,19.095502,6.0,2.0,4,1,"{19.095502,47.563829}","{19.111834,47.49461}",7.793884,2195,5.67
"""00e88bd3e0fcd94d5e4aa5aeb046b3…",1,2023-11-05 09:44:29 CET,2023-11-05 09:50:38 CET,6m 9s,47.24001,18.63222,0.0,0.0,2,3,"{18.63222,47.24001}","{18.844284,47.367169}",21.345006,3951,18.971
"""001db4a1248ca9db6d1cf538f0fca2…",1,2025-07-15 09:18:09 CEST,2025-07-15 10:09:34 CEST,51m 25s,47.43995,19.169805,1.0,6.0,8,2,"{19.169805,47.43995}","{19.121671,47.510319}",8.620419,871653,4.303
